In [1]:
import json
import pandas as pd
import numpy as np
from glob import glob

In [2]:
categories = ['Adherence','Concern','Education','Employment','Financial','Healthcare','Insurance','Literacy','Living','MentalHealth','Recommendation','Smoke','Social','SubstanceUse','Transportation','Trauma','overall']

In [3]:
def fix_bio_tags(bio_lists):
    fixed = []
    for tag_list in bio_lists:
        new_tags = []
        for i, tag in enumerate(tag_list):
            # If tag is 'O', nothing to change.
            if tag == 'O':
                new_tags.append(tag)
            else:
                prefix, entity = tag.split('-', 1)
                # If first token, or previous token is 'O', or previous token is a different entity,
                # then this token should be a beginning (B-) tag.
                if i == 0 or tag_list[i-1] == 'O' or (tag_list[i-1] != 'O' and tag_list[i-1].split('-', 1)[1] != entity):
                    new_tags.append('B-' + entity)
                else:
                    # Otherwise, continue the entity span as an inside (I-) tag.
                    new_tags.append('I-' + entity)
        fixed.append(new_tags)
    return fixed

def refine_bio_tags(bio_lists):
    refined = []
    for tags in bio_lists:
        new_tags = tags[:]  # work on a copy
        # Iterate from the second token to the second-to-last token.
        for i in range(1, len(new_tags) - 1):
            # Look for an "O" token.
            if new_tags[i] == 'O':
                prev_tag = new_tags[i - 1]
                next_tag = new_tags[i + 1]
                # Check if the previous token is part of an entity and the next token starts an entity.
                if prev_tag != 'O' and next_tag.startswith('B-'):
                    prev_entity = prev_tag.split('-', 1)[1]
                    next_entity = next_tag.split('-', 1)[1]
                    # If both tokens refer to the same entity, fill in the gap and adjust the following token.
                    if prev_entity == next_entity:
                        new_tags[i] = 'I-' + prev_entity
                        new_tags[i + 1] = 'I-' + next_entity
        refined.append(new_tags)
    return refined

In [4]:
def update_dict(y, cdict):
    for s in y:
        for tag in s:
            if tag.startswith('B-'):
                label = tag.split('-')[1]
                if label not in cdict.keys():
                    cdict[label] = 1
                else:
                    cdict[label] += 1
    return cdict
    

In [5]:
true_dict = {}
pred_dict = {}
for fold in range(1,6):
    for d in glob(f'../output/BERT*fold_{fold}*/predictions/'):
        y_true = []
        with open(f'../data/splitted_data/fold_{fold}/test.json','r') as textfile:
            for i in json.load(textfile):
                y_true.append(i['ner_tags'])
        overall_eval = {}
        predict_file = d + '/predictions.txt'
        output_file = d + '/evaluation.json'
        y_predict = []
        with open(predict_file) as txtfile:
            for i in txtfile.readlines():
                y_predict.append(i.split())
        y_predict = refine_bio_tags(fix_bio_tags(y_predict))   
        true_dict = update_dict(y_true, true_dict)
        pred_dict = update_dict(y_predict, pred_dict)
            

In [6]:
eval_dict = {i:
             {'strict': {'precision':0, 'recall':0, 'f-1':0}, 'relax': {'precision':0, 'recall':0, 'f-1':0}} for i in categories}
# for i in categories

In [7]:
for fold in range(1,6):
    for d in glob(f'../output/BERT*fold_{fold}*/predictions/'):
        metric = json.load(open(f'{d}evaluation.json', 'r'))
        for cat, vs in metric.items():
            if cat == 'sentence':
                continue
            for cri, vs2 in vs.items():
                for prf, v in vs2.items():
                    if v == 'N/A':
                        eval_dict[cat][cri][prf] += 0
                    elif v >= 0:
                        eval_dict[cat][cri][prf] += v/5
                    else:
                        eval_dict[cat][cri][prf] += 0

FileNotFoundError: [Errno 2] No such file or directory: '../output\\BERT_fold_1_lr_5e-5\\predictions\\evaluation.json'

In [8]:
flattened_data = []
for category, metrics in eval_dict.items():
    for condition, scores in metrics.items():
        flattened_data.append({'Category': category, 'Condition': condition, **scores})

# Create DataFrame
df = np.round(pd.DataFrame(flattened_data), 2)

In [15]:
df.to_excel('../output/category_eval_BERT.xlsx')

In [20]:
eval_dict

{'Adherence': {'strict': {'precision': 0.16,
   'recall': 0.2,
   'f-1': 0.17777777777777778},
  'relax': {'precision': 0.16, 'recall': 0.2, 'f-1': 0.17777777777777778}},
 'Alcohol': {'strict': {'precision': 0.2, 'recall': 0.2, 'f-1': 0.2},
  'relax': {'precision': 0.2, 'recall': 0.2, 'f-1': 0.2}},
 'Concern': {'strict': {'precision': 0.1,
   'recall': 0.05,
   'f-1': 0.06666666666666667},
  'relax': {'precision': 0.1, 'recall': 0.05, 'f-1': 0.06666666666666667}},
 'Drug': {'strict': {'precision': 0, 'recall': 0.0, 'f-1': 0.0},
  'relax': {'precision': 0, 'recall': 0.0, 'f-1': 0.0}},
 'Education': {'strict': {'precision': 0.12307692307692308,
   'recall': 0.14545454545454545,
   'f-1': 0.13333333333333336},
  'relax': {'precision': 0.12307692307692308,
   'recall': 0.14545454545454545,
   'f-1': 0.13333333333333336}},
 'Employment': {'strict': {'precision': 0.144,
   'recall': 0.1894736842105263,
   'f-1': 0.16363636363636364},
  'relax': {'precision': 0.144,
   'recall': 0.18947368421